# Virtual Cell MVP

This notebook builds a minimal bidirectional retrieval prototype over the bioRxiv marker extraction corpus.

A **profile** is one `(paper, cell type)` object with three linked parts:

- a cell-type label
- a set of marker genes
- the manuscript sentences that justify the marker claims

The notebook supports two query modes:

1. **Text query**: cosine similarity over a TF-IDF representation of `cell type label + evidence sentences`
2. **Gene query**: Jaccard similarity over the profile's marker-gene set, using either gene symbols or Ensembl IDs

This is a runnable MVP. It is not a trained multimodal model.

In [1]:
from __future__ import annotations

import re
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def find_repo_root() -> Path:
    here = Path.cwd()
    candidates = [here, here.parent]
    for root in candidates:
        if (root / 'docs' / 'llmarkers.sqlite').exists():
            return root
    raise FileNotFoundError('Could not locate docs/llmarkers.sqlite from the current working directory.')


REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / 'docs' / 'llmarkers.sqlite'
print(f'Repo root: {REPO_ROOT}')
print(f'Database:  {DB_PATH}')

Repo root: /Users/sinabooeshaghi/projects/markergeneextraction/llmarkers
Database:  /Users/sinabooeshaghi/projects/markergeneextraction/llmarkers/docs/llmarkers.sqlite


In [2]:
query = '''
SELECT
    p.paper_id,
    p.doi,
    p.title,
    p.year,
    p.license,
    m.group_name,
    m.feature_name,
    m.feature_id,
    m.source_rationale,
    m.source_type
FROM papers AS p
JOIN markers AS m ON m.paper_id = p.paper_id
WHERE COALESCE(m.data_id, '') = ''
  AND COALESCE(m.group_name, '') <> ''
  AND COALESCE(m.feature_name, '') <> ''
ORDER BY p.paper_id, m.group_name, m.feature_name
'''

with sqlite3.connect(DB_PATH) as conn:
    rows = pd.read_sql_query(query, conn)

print(f'Loaded {len(rows):,} bioRxiv marker rows from SQLite.')
display(rows.head())

Loaded 12,516 bioRxiv marker rows from SQLite.


,paper_id,doi,title,year,license,group_name,feature_name,feature_id,source_rationale,source_type
0,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,BEST1,ENSG00000167995,marker shows sign of expression in figure,image
1,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,BEST1,ENSG00000167995,marker shows sign of expression in figure,image
2,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,CNGB3,ENSG00000170289,marker shows sign of expression in figure,image
3,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,COL4A3,ENSG00000169031,marker shows sign of expression in figure,image
4,4,10.1038/s41467-021-25968-8,Multi-species single-cell transcriptomic analy...,2021,NaN,RETINAL PIGMENTED EPITHELIUM,CRX,ENSG00000105392,marker shows sign of expression in figure,image


In [3]:
def clean_text(value: str | None) -> str:
    if value is None:
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()


def dedupe_keep_order(values):
    out = []
    seen = set()
    for value in values:
        if not value:
            continue
        if value in seen:
            continue
        seen.add(value)
        out.append(value)
    return out


def normalize_doi(value: str | None) -> str:
    if not value:
        return ''
    text = clean_text(value).lower()
    for prefix in ('https://doi.org/', 'http://doi.org/', 'doi:'):
        if text.startswith(prefix):
            text = text[len(prefix):]
            break
    return text.rstrip('. )]')


def extract_abstracts_from_meca(root: Path) -> dict[str, str]:
    context = {}
    meca_root = root / 'data' / 'biorxiv' / 'meca'
    for folder in meca_root.iterdir():
        if not folder.is_dir():
            continue
        xml_paths = [
            p for p in folder.glob('*.xml')
            if p.name not in {'directives.xml', 'manifest.xml', 'transfer.xml'}
        ]
        if not xml_paths:
            continue
        xml_path = xml_paths[0]
        try:
            root_xml = ET.parse(xml_path).getroot()
        except ET.ParseError:
            continue

        doi = ''
        for node in root_xml.findall('.//article-id'):
            if node.attrib.get('pub-id-type') == 'doi' and node.text:
                doi = normalize_doi(node.text)
                if doi:
                    break
        if not doi:
            continue

        abstract_node = root_xml.find('.//abstract')
        if abstract_node is None:
            continue
        abstract_text = clean_text(' '.join(abstract_node.itertext()))
        if abstract_text:
            context[doi] = abstract_text
    return context


paper_context_by_doi = extract_abstracts_from_meca(REPO_ROOT)
print(f'Loaded title/abstract context for {len(paper_context_by_doi):,} papers from MECA XML.')


records = []
group_cols = ['paper_id', 'doi', 'title', 'year', 'license', 'group_name']
for key, grp in rows.groupby(group_cols, dropna=False, sort=False):
    paper_id, doi, title, year, license_text, group_name = key
    gene_names = dedupe_keep_order(clean_text(v).upper() for v in grp['feature_name'].tolist())
    gene_ids = dedupe_keep_order(clean_text(v).upper() for v in grp['feature_id'].fillna('').tolist())
    sentences = dedupe_keep_order(clean_text(v) for v in grp['source_rationale'].fillna('').tolist())
    label = clean_text(group_name)
    doi_norm = normalize_doi(doi)
    paper_context_blob = ' '.join(part for part in [clean_text(title), paper_context_by_doi.get(doi_norm, '')] if part)
    # Repeat the label to keep exact cell-type wording important during retrieval.
    text_blob = ' '.join([label, label, label] + sentences)
    records.append(
        {
            'profile_id': f'{paper_id}::{clean_text(group_name)}',
            'paper_id': int(paper_id),
            'doi': clean_text(doi),
            'title': clean_text(title),
            'year': int(year) if pd.notna(year) else None,
            'license': clean_text(license_text),
            'cell_type_label': clean_text(group_name),
            'text_blob': text_blob,
            'paper_context_blob': paper_context_blob,
            'text_with_context': ' '.join(part for part in [text_blob, paper_context_blob] if part),
            'evidence_sentences': sentences,
            'gene_names': gene_names,
            'gene_ids': gene_ids,
            'n_genes': len(gene_names),
            'n_sentences': len(sentences),
        }
    )

profiles = pd.DataFrame.from_records(records)
profiles = profiles[profiles['n_genes'] > 0].reset_index(drop=True)

print(f'Built {len(profiles):,} paper-celltype profiles.')
summary = pd.DataFrame(
    {
        'stat': ['papers', 'profiles', 'papers with abstract context', 'median genes/profile', 'median evidence sentences/profile'],
        'value': [
            profiles['paper_id'].nunique(),
            len(profiles),
            int((profiles['paper_context_blob'].str.len() > 0).sum()),
            int(profiles['n_genes'].median()),
            int(profiles['n_sentences'].median()),
        ],
    }
)
display(summary)
display(
    profiles[['cell_type_label', 'n_genes', 'n_sentences', 'title']]
    .sort_values(['n_genes', 'n_sentences'], ascending=False)
    .head(10)
)


Loaded title/abstract context for 504 papers from MECA XML.


Built 3,729 paper-celltype profiles.


,stat,value
0,papers,422
1,profiles,3729
2,papers with abstract context,3729
3,median genes/profile,2
4,median evidence sentences/profile,1


,cell_type_label,n_genes,n_sentences,title
1673,EPI,61,21,"Single-cell transcriptome analysis of human, m..."
1221,SMC8-MAC,51,8,Macrophage-like vascular smooth muscle cells d...
1757,AMD/RPD−,48,7,Patient induced pluripotent stem cells identif...
1675,PRE,42,15,"Single-cell transcriptome analysis of human, m..."
1920,NEUTROPHILS,41,25,Bronchoalveolar Lavage Single-Cell Transcripto...
2707,CD4+ NAÏVE,39,12,Immunological Differences in Atopic Dermatitis...
277,VESTIBULAR HC,38,10,The Dual Molecular Identity of Vestibular Kino...
2472,CD8+ T CELL,36,6,Antigen specificity of clonally-enriched CD8+ ...
1756,AMD/RPD+,35,10,Patient induced pluripotent stem cells identif...
276,TYPE II HC,35,7,The Dual Molecular Identity of Vestibular Kino...


## Retrieval setup

The profile is the bridge object.

- **Text query** compares the user's phrase to the profile's `cell_type_label + evidence_sentences`
- **Gene query** compares the user's gene list to the profile's `gene_names` and `gene_ids`

Both queries return the same kind of result: the nearest profiles from the literature.

In [4]:
def normalize_profile_text(text: str) -> str:
    text = str(text).upper()
    text = text.replace('+', ' PLUS ').replace('-', ' MINUS ')
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


vectorizer = TfidfVectorizer(
    preprocessor=normalize_profile_text,
    token_pattern=r'(?u)\b[A-Z0-9]+\b',
    ngram_range=(1, 2),
    min_df=2,
)
text_matrix = vectorizer.fit_transform(profiles['text_blob'])
context_vectorizer = TfidfVectorizer(
    preprocessor=normalize_profile_text,
    token_pattern=r'(?u)\b[A-Z0-9]+\b',
    ngram_range=(1, 2),
    min_df=2,
)
context_matrix = context_vectorizer.fit_transform(profiles['paper_context_blob'])
print(f'Text vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Context vocabulary size: {len(context_vectorizer.vocabulary_):,}')

DEFAULT_PROFILE_WEIGHT = 0.9
DEFAULT_CONTEXT_WEIGHT = 0.1
print(f'Default hybrid weights: profile={DEFAULT_PROFILE_WEIGHT:.2f}, context={DEFAULT_CONTEXT_WEIGHT:.2f}')


def preview_sentence(sentences: list[str], max_len: int = 160) -> str:
    if not sentences:
        return ''
    text = sentences[0]
    return text if len(text) <= max_len else text[: max_len - 3] + '...'


def query_text(query_text: str, top_k: int = 8, mode: str = 'profile', profile_weight: float = DEFAULT_PROFILE_WEIGHT, context_weight: float = DEFAULT_CONTEXT_WEIGHT) -> pd.DataFrame:
    query_vec = vectorizer.transform([query_text])
    profile_scores = cosine_similarity(query_vec, text_matrix).ravel()
    context_query_vec = context_vectorizer.transform([query_text])
    context_scores = cosine_similarity(context_query_vec, context_matrix).ravel()
    if mode == 'context':
        scores = context_scores
    elif mode == 'hybrid':
        scores = profile_weight * profile_scores + context_weight * context_scores
    else:
        scores = profile_scores
    order = np.argsort(scores)[::-1][:top_k]
    result = profiles.iloc[order][['cell_type_label', 'title', 'doi', 'n_genes', 'n_sentences']].copy()
    result.insert(0, 'score', scores[order])
    result['profile_score'] = profile_scores[order]
    result['context_score'] = context_scores[order]
    result['top_genes'] = [', '.join(profiles.iloc[i]['gene_names'][:6]) for i in order]
    result['evidence_preview'] = [preview_sentence(profiles.iloc[i]['evidence_sentences']) for i in order]
    return result.reset_index(drop=True)


Text vocabulary size: 38,007
Context vocabulary size: 66,244
Default hybrid weights: profile=0.90, context=0.10


In [5]:
text_queries = [
    't cells',
    'cytotoxic lymphocytes',
    'fibroblasts in stromal tissue',
]

for q in text_queries:
    print(f'\nTEXT QUERY: {q}')
    display(query_text(q, top_k=5))


TEXT QUERY: t cells


,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,0.472143,T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,5,1,0.472143,0.008620,"CD247, CD3D, CD3E, CD3G, ZAP70",T cells expressed pan-T cell marker CD3E and o...
1,0.456107,CYTOTOXIC ΑΒ T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,2,1,0.456107,0.008620,"GNLY, GZMB","In spleen, two populations of non-cycling αβ T..."
2,0.440956,CD4+ T CELL,Dual inhibition of the nonsense mediated mRNA ...,10.64898/2025.12.04.692418,1,1,0.440956,0.047109,CD4,"Labels include: CD4+ T cells (CD4+T), regulato..."
3,0.435409,NK CELL,Deciphering Tumor Microenvironment Dynamics in...,10.1101/2025.11.27.690370,2,1,0.435409,0.141686,"KIR2DL4, NCAM1",These subgroups encompassed CD4+ and CD8+ naïv...
4,0.420827,NAÏVE T CELLS/STEM CELL MEMORY T CELLS,A Tonic Signaling Code Predicts CAR-T Cell Eff...,10.1101/2025.09.29.679095,2,1,0.420827,0.074448,"CD45RO, CD62L",Surface expression of CD62L and CD45RO was use...



TEXT QUERY: cytotoxic lymphocytes


,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,0.299679,CYTOTOXIC PLASMA CELL,Revised International Staging System (R-ISS) s...,10.1101/2021.12.06.471423,5,1,0.299679,0.058048,"CCL4, CCL5, GNLY, GZMA, NKG7","The cytotoxic genes NKG7, GZMA, and GNLY and t..."
1,0.269802,CYTOTOXIC CD8+ T CELL,Multi-omics analysis reveals key immunogenic s...,10.1101/2024.11.28.625843,1,1,0.269802,0.013309,GZMB,Upregulation of the principal cytotoxic granzy...
2,0.260614,LYMPHOCYTES,MAVS mediates a protective immune response in ...,10.1101/2021.12.22.473954,6,1,0.260614,0.000000,"GZMB, IFI44, IFNG, IRF7, PRF1, XCL1",Upon examining genes that were differentially ...
3,0.246870,CYTOTOXIC,NK cells contribute to resistance to anti-PD1 ...,10.1101/2023.12.14.571631,3,1,0.246870,0.000000,"GNLY, GZMB, PRF1","In parallel, re-clustering identified four CD8..."
4,0.240097,PRE-CYTOTOXIC PHENOTYPE,Single cell profiling reveals functional heter...,10.1101/2022.01.24.477494,2,2,0.240097,0.031204,"CD107A, NKP46","Notably, no difference in CD107a expression up..."



TEXT QUERY: fibroblasts in stromal tissue


,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,0.170454,STROMAL CELL,Endocrine disruption of early uterine differen...,10.1101/2022.11.18.517135,4,2,0.170454,0.018620,"COL6A3, DPT, FOXL2, VCAN","Feature plots of three stromal cell markers, D..."
1,0.137103,FIBROBLAST,"SenSet, a novel human lung senescence cell gen...",10.1101/2024.12.21.629928,25,10,0.137103,0.008235,"CALR, CITED2, CNN2, GAPDH, GUK1, ID2","Notably, 15 genes downregulated in fibroblasts..."
2,0.136187,FIBROBLAST,Metastatic prognostic ability of lung cancer s...,10.1101/2023.07.10.548323,1,1,0.136187,0.094336,DCN,"In addition, DCN was used as a cell-type marke..."
3,0.132119,STROMAL CELL,Spatial and Single-Cell Transcriptomics Deciph...,10.1101/2025.11.22.689892,1,1,0.132119,0.002769,MGP,"Expression patterns of PTPRC, MGP, and EPCAM d..."
4,0.125392,TISSUE STEM CELL,Revealing the Organ-Specific Expression of SPT...,10.1101/2023.06.01.543198,2,2,0.125392,0.042316,"EPDR1, SPTBN1",EPDR1 is expressed in the Tissue stem cells of...


## Adding paper context: title + abstract

The baseline text search above uses only the profile-local signal: `cell type label + evidence sentences`.

Here we add a second signal from the **paper title + abstract** and compare:

- `profile`: profile-local text only
- `hybrid`: `0.7 * profile + 0.3 * paper context`

This should help broader contextual queries such as tissue, disease, or assay setting, while preserving the cell-type signal.

In [6]:
context_queries = [
    'tumor infiltrating t cells',
    'fibroblasts in stromal tissue',
    'immune cells in lung',
]

for q in context_queries:
    print(f'\nCOMPARISON QUERY: {q}')
    baseline = query_text(q, top_k=5, mode='profile').copy()
    hybrid = query_text(q, top_k=5, mode='hybrid').copy()
    baseline.insert(0, 'mode', 'profile')
    hybrid.insert(0, 'mode', 'hybrid')
    display(pd.concat([baseline, hybrid], ignore_index=True))


COMPARISON QUERY: tumor infiltrating t cells


,mode,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,profile,0.291303,CD4+ T CELL,Dual inhibition of the nonsense mediated mRNA ...,10.64898/2025.12.04.692418,1,1,0.291303,0.024246,CD4,"Labels include: CD4+ T cells (CD4+T), regulato..."
1,profile,0.290957,TUMOR,signifinder enables the identification of tumo...,10.1101/2023.03.07.530940,1,1,0.290957,0.025930,PD-L1,"In ductal breast carcinoma, the presence of a ..."
2,profile,0.220197,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...,The CD8+ T cell landscape of human brain metas...,10.1101/2021.08.03.455000,3,1,0.220197,0.243678,"IL7R, TCF7, TOX","Importantly, some of these experimentally-vali..."
3,profile,0.215639,TUMOR CELL,Massively parallel single-cell chromatin lands...,10.1101/610550,1,1,0.215639,0.049118,GLI1,"Moreover, tumor cells showed high accessibilit..."
4,profile,0.210628,T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,5,1,0.210628,0.004437,"CD247, CD3D, CD3E, CD3G, ZAP70",T cells expressed pan-T cell marker CD3E and o...
5,hybrid,0.264597,CD4+ T CELL,Dual inhibition of the nonsense mediated mRNA ...,10.64898/2025.12.04.692418,1,1,0.291303,0.024246,CD4,"Labels include: CD4+ T cells (CD4+T), regulato..."
6,hybrid,0.264454,TUMOR,signifinder enables the identification of tumo...,10.1101/2023.03.07.530940,1,1,0.290957,0.025930,PD-L1,"In ductal breast carcinoma, the presence of a ..."
7,hybrid,0.222545,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...,The CD8+ T cell landscape of human brain metas...,10.1101/2021.08.03.455000,3,1,0.220197,0.243678,"IL7R, TCF7, TOX","Importantly, some of these experimentally-vali..."
8,hybrid,0.198987,TUMOR CELL,Massively parallel single-cell chromatin lands...,10.1101/610550,1,1,0.215639,0.049118,GLI1,"Moreover, tumor cells showed high accessibilit..."
9,hybrid,0.190009,T CELLS,A single-cell immune atlas of primary and seco...,10.1101/2025.09.11.675613,5,1,0.210628,0.004437,"CD247, CD3D, CD3E, CD3G, ZAP70",T cells expressed pan-T cell marker CD3E and o...



COMPARISON QUERY: fibroblasts in stromal tissue


,mode,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,profile,0.170454,STROMAL CELL,Endocrine disruption of early uterine differen...,10.1101/2022.11.18.517135,4,2,0.170454,0.018620,"COL6A3, DPT, FOXL2, VCAN","Feature plots of three stromal cell markers, D..."
1,profile,0.137103,FIBROBLAST,"SenSet, a novel human lung senescence cell gen...",10.1101/2024.12.21.629928,25,10,0.137103,0.008235,"CALR, CITED2, CNN2, GAPDH, GUK1, ID2","Notably, 15 genes downregulated in fibroblasts..."
2,profile,0.136187,FIBROBLAST,Metastatic prognostic ability of lung cancer s...,10.1101/2023.07.10.548323,1,1,0.136187,0.094336,DCN,"In addition, DCN was used as a cell-type marke..."
3,profile,0.132119,STROMAL CELL,Spatial and Single-Cell Transcriptomics Deciph...,10.1101/2025.11.22.689892,1,1,0.132119,0.002769,MGP,"Expression patterns of PTPRC, MGP, and EPCAM d..."
4,profile,0.125392,TISSUE STEM CELL,Revealing the Organ-Specific Expression of SPT...,10.1101/2023.06.01.543198,2,2,0.125392,0.042316,"EPDR1, SPTBN1",EPDR1 is expressed in the Tissue stem cells of...
5,hybrid,0.155271,STROMAL CELL,Endocrine disruption of early uterine differen...,10.1101/2022.11.18.517135,4,2,0.170454,0.018620,"COL6A3, DPT, FOXL2, VCAN","Feature plots of three stromal cell markers, D..."
6,hybrid,0.132002,FIBROBLAST,Metastatic prognostic ability of lung cancer s...,10.1101/2023.07.10.548323,1,1,0.136187,0.094336,DCN,"In addition, DCN was used as a cell-type marke..."
7,hybrid,0.124216,FIBROBLAST,"SenSet, a novel human lung senescence cell gen...",10.1101/2024.12.21.629928,25,10,0.137103,0.008235,"CALR, CITED2, CNN2, GAPDH, GUK1, ID2","Notably, 15 genes downregulated in fibroblasts..."
8,hybrid,0.119184,STROMAL CELL,Spatial and Single-Cell Transcriptomics Deciph...,10.1101/2025.11.22.689892,1,1,0.132119,0.002769,MGP,"Expression patterns of PTPRC, MGP, and EPCAM d..."
9,hybrid,0.117084,TISSUE STEM CELL,Revealing the Organ-Specific Expression of SPT...,10.1101/2023.06.01.543198,2,2,0.125392,0.042316,"EPDR1, SPTBN1",EPDR1 is expressed in the Tissue stem cells of...



COMPARISON QUERY: immune cells in lung


,mode,score,cell_type_label,title,doi,n_genes,n_sentences,profile_score,context_score,top_genes,evidence_preview
0,profile,0.247188,LUNG EPITHELIAL CELL,eSPred: Explainable scRNA-seq Prediction via C...,10.1101/2025.05.14.654052,1,1,0.247188,0.000000,ACE2,SARS-CoV-2 infection disrupts these processes ...
1,profile,0.244607,LUNG ADENOCARCINOMA,eSPred: Explainable scRNA-seq Prediction via C...,10.1101/2025.05.14.654052,1,1,0.244607,0.000000,HMMR,"In lung adenocarcinoma (LUAD), overexpression ..."
2,profile,0.197849,AT2,SARS-CoV-2 receptor ACE2 and TMPRSS2 are predo...,10.1101/2020.03.13.991455,3,3,0.197849,0.060415,"ACE2, FURIN, TMPRSS2","As expected from prior literature, the AT2 cel..."
3,profile,0.185782,IMMUNE CELL,Flexible and robust cell type annotation for h...,10.1101/2024.09.12.612510,1,1,0.185782,0.004892,CD45,Note that CD45 in the tissue structure and ner...
4,profile,0.179279,IMMUNE CELL,Cryobanking of human distal lung epithelial ce...,10.1101/2021.11.15.468402,1,1,0.179279,0.065728,CD45,Non-epithelial cell types including immune cel...
5,hybrid,0.222469,LUNG EPITHELIAL CELL,eSPred: Explainable scRNA-seq Prediction via C...,10.1101/2025.05.14.654052,1,1,0.247188,0.000000,ACE2,SARS-CoV-2 infection disrupts these processes ...
6,hybrid,0.220146,LUNG ADENOCARCINOMA,eSPred: Explainable scRNA-seq Prediction via C...,10.1101/2025.05.14.654052,1,1,0.244607,0.000000,HMMR,"In lung adenocarcinoma (LUAD), overexpression ..."
7,hybrid,0.184106,AT2,SARS-CoV-2 receptor ACE2 and TMPRSS2 are predo...,10.1101/2020.03.13.991455,3,3,0.197849,0.060415,"ACE2, FURIN, TMPRSS2","As expected from prior literature, the AT2 cel..."
8,hybrid,0.167924,IMMUNE CELL,Cryobanking of human distal lung epithelial ce...,10.1101/2021.11.15.468402,1,1,0.179279,0.065728,CD45,Non-epithelial cell types including immune cel...
9,hybrid,0.167693,IMMUNE CELL,Flexible and robust cell type annotation for h...,10.1101/2024.09.12.612510,1,1,0.185782,0.004892,CD45,Note that CD45 in the tissue structure and ner...


## Tuning the hybrid weights

The hybrid search has one simple parameter pair:

- `profile_weight`: how much to trust the cell-type label + evidence sentences
- `context_weight`: how much to trust the paper title + abstract

I sweep a small grid below and inspect the top 3 results for each query. The goal is pragmatic rather than formal optimization: choose weights that add useful paper context without washing out the profile-local cell-type signal.

In [7]:
weight_grid = [
    (1.00, 0.00),
    (0.95, 0.05),
    (0.90, 0.10),
    (0.85, 0.15),
    (0.80, 0.20),
    (0.75, 0.25),
]

sweep_queries = [
    'tumor infiltrating t cells',
    'fibroblasts in stromal tissue',
    'immune cells in lung',
    'activated monocytes in blood',
    'brain metastasis t cells',
]

rows = []
for q in sweep_queries:
    for pw, cw in weight_grid:
        result = query_text(q, top_k=3, mode='hybrid', profile_weight=pw, context_weight=cw)
        rows.append(
            {
                'query': q,
                'profile_weight': pw,
                'context_weight': cw,
                'top_1': result.loc[0, 'cell_type_label'],
                'top_2': result.loc[1, 'cell_type_label'],
                'top_3': result.loc[2, 'cell_type_label'],
            }
        )

weight_sweep_df = pd.DataFrame(rows)
display(weight_sweep_df)


,query,profile_weight,context_weight,top_1,top_2,top_3
0,tumor infiltrating t cells,1.00,0.00,CD4+ T CELL,TUMOR,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...
1,tumor infiltrating t cells,0.95,0.05,CD4+ T CELL,TUMOR,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...
2,tumor infiltrating t cells,0.90,0.10,CD4+ T CELL,TUMOR,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...
3,tumor infiltrating t cells,0.85,0.15,CD4+ T CELL,TUMOR,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...
4,tumor infiltrating t cells,0.80,0.20,TUMOR,CD4+ T CELL,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...
5,tumor infiltrating t cells,0.75,0.25,CEFX-SPECIFIC BRAIN METASTASIS-INFILTRATING PD...,TUMOR,CD4+ T CELL
6,fibroblasts in stromal tissue,1.00,0.00,STROMAL CELL,FIBROBLAST,FIBROBLAST
7,fibroblasts in stromal tissue,0.95,0.05,STROMAL CELL,FIBROBLAST,FIBROBLAST
8,fibroblasts in stromal tissue,0.90,0.10,STROMAL CELL,FIBROBLAST,FIBROBLAST
9,fibroblasts in stromal tissue,0.85,0.15,STROMAL CELL,FIBROBLAST,FIBROBLAST


### Selected default

From the sweep above, the most sensible default is:

- `profile_weight = 0.90`
- `context_weight = 0.10`

Reasoning:

- It preserves the profile-local cell-type signal for straightforward biological queries.
- It adds some paper-level context without introducing many obvious context-driven false positives.
- Heavier context weighting (`0.20` to `0.25`) helps some broad queries, but it also starts to pull in more off-target labels such as generic activated states.

So `0.90 / 0.10` is a good website default. If the goal later becomes more contextual search, it would be reasonable to expose the weights as an advanced control.

In [8]:
profiles['gene_name_set'] = profiles['gene_names'].map(set)
profiles['gene_id_set'] = profiles['gene_ids'].map(lambda ids: {g for g in ids if g})
profiles['gene_token_set'] = profiles.apply(lambda row: row['gene_name_set'] | row['gene_id_set'], axis=1)


def parse_gene_query(query: str) -> set[str]:
    tokens = re.split(r'[,;\s]+', query.upper().strip())
    return {token for token in tokens if token}


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)


def query_genes(query: str, top_k: int = 8) -> pd.DataFrame:
    query_set = parse_gene_query(query)
    scores = profiles['gene_token_set'].map(lambda genes: jaccard(query_set, genes)).to_numpy()
    order = np.argsort(scores)[::-1][:top_k]
    result = profiles.iloc[order][['cell_type_label', 'title', 'doi', 'n_genes', 'n_sentences']].copy()
    result.insert(0, 'jaccard', scores[order])
    result['query_genes'] = ', '.join(sorted(query_set))
    result['top_genes'] = [', '.join(profiles.iloc[i]['gene_names'][:8]) for i in order]
    result['shared_matches'] = [', '.join(sorted(query_set & profiles.iloc[i]['gene_token_set'])) for i in order]
    result['evidence_preview'] = [preview_sentence(profiles.iloc[i]['evidence_sentences']) for i in order]
    return result.reset_index(drop=True)


In [9]:
gene_queries = [
    'IL7R LTB MALAT1',
    'ENSG00000168685',
    'ENSG00000105374 ENSG00000180644 ENSG00000115523',
    'DCN COL1A1 COL1A2',
]

for q in gene_queries:
    print(f'\nGENE QUERY: {q}')
    display(query_genes(q, top_k=5))


GENE QUERY: IL7R LTB MALAT1


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.333333,PRE-B,Stabilized marker gene identification and func...,10.1101/2024.08.21.608838,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,"Importantly, scSCOPE accurately identified Il7..."
1,0.333333,B-CELL,Stabilized marker gene identification and func...,10.1101/2024.08.21.608838,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,"As an example, scSCOPE identified Il7r, a gene..."
2,0.333333,LATE-PRO,Stabilized marker gene identification and func...,10.1101/2024.08.21.608838,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,"Importantly, scSCOPE accurately identified Il7..."
3,0.250000,IL7R+CD8_T,Tracing the stemness and malignant transition ...,10.1101/2025.04.06.647495,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,Six major T cell subtypes were identified: Naï...
4,0.250000,CD8-C3-IL7R,Single-cell atlas of tumor clonal evolution in...,10.1101/2020.08.18.254748,1,1,"IL7R, LTB, MALAT1",IL7R,IL7R,Large proportions of memory T cells of CD4-c2-...



GENE QUERY: ENSG00000168685


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.5,IL7RHIGHCD8+T,Tracing the stemness and malignant transition ...,10.1101/2025.04.06.647495,1,1,ENSG00000168685,IL7R,ENSG00000168685,Pseudotime trajectory analysis revealed T cell...
1,0.5,ILC1,Big data and single cell transcriptomics: impl...,10.1101/257352,1,1,ENSG00000168685,CD127,ENSG00000168685,Bjorklund et al. used scRNAseq to explore the ...
2,0.5,CD4 T CELL,Accurate highly variable gene selection using ...,10.1101/2025.06.23.661026,1,1,ENSG00000168685,IL7R,ENSG00000168685,"These include, for example, IL7R (CD4 T cells)..."
3,0.5,ILC3,Big data and single cell transcriptomics: impl...,10.1101/257352,1,1,ENSG00000168685,CD127,ENSG00000168685,Bjorklund et al. used scRNAseq to explore the ...
4,0.5,CD4+ T CELL,PBMC Fixation and Processing for Chromium Sing...,10.1101/315267,1,1,ENSG00000168685,IL7R,ENSG00000168685,Finer substructures were detected with the T-c...



GENE QUERY: ENSG00000105374 ENSG00000180644 ENSG00000115523


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.400000,NK-CELL GROUP,Decontamination of ambient RNA in single-cell ...,10.1101/704015,2,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","GNLY, NKG7","ENSG00000105374, ENSG00000115523",Cell clusters 2 and 3 were classified as B-cel...
1,0.400000,NK CELL,Comparative Analysis of Feature Selection Meth...,10.64898/2025.12.02.691907,2,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","GNLY, NKG7","ENSG00000105374, ENSG00000115523",Known marker recovery was evaluated for the PB...
2,0.300000,CD8.5,Acquisition of discrete immune suppressive bar...,10.1101/2024.12.31.630523,5,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","FCGR3A, FGFBP2, GNLY, NKG7, PRF1","ENSG00000105374, ENSG00000115523, ENSG00000180644",Clusters CD8.3 and CD8.5 with increased expres...
3,0.300000,CD8.3,Acquisition of discrete immune suppressive bar...,10.1101/2024.12.31.630523,5,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","FCGR3A, FGFBP2, GNLY, NKG7, PRF1","ENSG00000105374, ENSG00000115523, ENSG00000180644",Clusters CD8.3 and CD8.5 with increased expres...
4,0.285714,CD8+ T CELL,Single-cell RNA sequencing of peripheral blood...,10.1101/2024.01.04.574155,3,1,"ENSG00000105374, ENSG00000115523, ENSG00000180644","GNLY, GZMK, NKG7","ENSG00000105374, ENSG00000115523","In contrast, CD4+ cells expressed markedly hig..."



GENE QUERY: DCN COL1A1 COL1A2


,jaccard,cell_type_label,title,doi,n_genes,n_sentences,query_genes,top_genes,shared_matches,evidence_preview
0,0.5,MESENCHYMAL CELL,A comprehensive atlas of testicular interstiti...,10.1101/2024.08.02.606288,3,1,"COL1A1, COL1A2, DCN","COL1A2, COL3A1, DCN","COL1A2, DCN",Through uniform manifold approximation and pro...
1,0.5,FB,Single cell transcriptome analysis of cavernou...,10.1101/2023.05.25.542355,3,1,"COL1A1, COL1A2, DCN","COL1A1, COL1A2, FBLN1","COL1A1, COL1A2",The clusters were annotated by expression of m...
2,0.4,MKI67-MCS,A comprehensive atlas of testicular interstiti...,10.1101/2024.08.02.606288,4,1,"COL1A1, COL1A2, DCN","COL1A2, DCN, HELLS, LIG1","COL1A2, DCN","Additionally, we identified two proliferating ..."
3,0.4,FIBROBLAST,A multi-gene predictive model for the radiatio...,10.1101/2024.06.10.598247,2,1,"COL1A1, COL1A2, DCN","COL1A1, DCN","COL1A1, DCN",Fibroblasts expressed DCN and COL1A1 genes.
4,0.4,FIBROBLAST,Deciphering Tumor Microenvironment Dynamics in...,10.1101/2025.11.27.690370,2,1,"COL1A1, COL1A2, DCN","COL1A1, DCN","COL1A1, DCN",Annotation was supported by the expression of ...


## What this MVP shows

This notebook demonstrates a pragmatic profile-centric virtual cell prototype:

- natural-language cell-type descriptions can retrieve literature marker profiles
- gene lists can retrieve the same profiles in the reverse direction
- each result keeps provenance through the paper title, DOI, and evidence sentence
- adding title + abstract as a secondary signal can help broader contextual queries, but the profile-local text remains the core retrieval object

A stronger version could replace TF-IDF with a semantic text embedding model and add a learned cross-modal alignment between text and gene-profile representations.